In [3]:
import pandas as pd

files = {
    2019: r"C:\...\2300_2019.csv",
    2020: r"C:\...\2300_2020.csv",
    2021: r"C:\...\2300_2021.csv"
}

# Порядок показателей (как в исходном файле)
indicator_order = [
    "Взято на учет рецидивов",
    "Взято на учет рецидивов III группы",
    "Прибыло",
    "Переведено в III группу",
    "Диагноз туберкулеза снят",
    "Выбыло",
    "Умерло от туберкулеза",
    "Умерло от других причин"
]

# Сопоставление показателей (с учетом возможных колонок)
indicators_by_name = {
    "Взято на учет рецидивов": "Взято на учет рецидивов",
    "    из них из Ш группы": "Взято на учет рецидивов III группы",
    "из них из Ш группы": "Взято на учет рецидивов III группы",
    "Прибыло": "Прибыло",
    "Переведено в Ш группу": "Переведено в III группу",
    "Диагноз туберкулеза снят": "Диагноз туберкулеза снят",
    "Выбыло": "Выбыло",
    "Умерло от туберкулеза": "Умерло от туберкулеза",
    "Умерло от других причин": "Умерло от других причин"
}

# Все возможные комбинации (Показатель, Уточнение, Еще уточнение)
categories = [
    ("Туберкулез органов дыхания", "Всего"),
    ("Туберкулез органов дыхания", "Дети от 0 до 14 лет"),
    ("Туберкулез органов дыхания", "Подростки от 15 до 17 лет"),
    ("Туберкулез органов дыхания из них туберкулез легких", "Всего"),
    ("Туберкулез органов дыхания из них туберкулез легких", "Дети от 0 до 14 лет"),
    ("Туберкулез органов дыхания из них туберкулез легких", "Подростки от 15 до 17 лет"),
    ("Прочие формы туберкулеза", "Всего"),
    ("Прочие формы туберкулеза", "Дети от 0 до 14 лет"),
    ("Прочие формы туберкулеза", "Подростки от 15 до 17 лет")
]

# Соответствие колонок с правильными номерами граф
# col_idx - номер колонки в DataFrame (начинается с 0)
# graph_number - номер графы в исходном документе (начинается с 3)
category_col_map = {
    ("Туберкулез органов дыхания", "Всего"): {"col_idx": 4, "graph_number": 3},
    ("Туберкулез органов дыхания", "Дети от 0 до 14 лет"): {"col_idx": 5, "graph_number": 4},
    ("Туберкулез органов дыхания", "Подростки от 15 до 17 лет"): {"col_idx": 6, "graph_number": 5},
    ("Туберкулез органов дыхания из них туберкулез легких", "Всего"): {"col_idx": 7, "graph_number": 6},
    ("Туберкулез органов дыхания из них туберкулез легких", "Дети от 0 до 14 лет"): {"col_idx": 8, "graph_number": 7},
    ("Туберкулез органов дыхания из них туберкулез легких", "Подростки от 15 до 17 лет"): {"col_idx": 9, "graph_number": 8},
    ("Прочие формы туберкулеза", "Всего"): {"col_idx": 10, "graph_number": 9},
    ("Прочие формы туберкулеза", "Дети от 0 до 14 лет"): {"col_idx": 11, "graph_number": 10},
    ("Прочие формы туберкулеза", "Подростки от 15 до 17 лет"): {"col_idx": 12, "graph_number": 11}
}

all_data = []

for year, filename in files.items():
    print(f"\nОбработка {year} года...")
    
    # Читаем файл
    df_raw = pd.read_csv(filename, header=None, encoding='utf-8')
    
    # Словарь для значений
    values_dict = {}
    row_number_dict = {}
    
    for idx, row in df_raw.iterrows():
        # Проверяем колонку 0 и колонку 2 для названия показателя
        indicator_name = ""
        if pd.notna(row[0]) and str(row[0]).strip():
            indicator_name = str(row[0]).strip()
        elif pd.notna(row[2]) and str(row[2]).strip():
            indicator_name = str(row[2]).strip()
        
        indicator_name = indicator_name.strip()
        
        # Проверяем соответствие
        matched_indicator = None
        for key in indicators_by_name:
            if key in indicator_name or indicator_name == key:
                matched_indicator = indicators_by_name[key]
                break
        
        if matched_indicator:
            print(f"  Найден показатель: '{matched_indicator}'")
            
            # Сохраняем номер строки из колонки 3
            row_number = None
            if pd.notna(row[3]) and str(row[3]).strip():
                try:
                    row_number = int(float(row[3]))
                except:
                    row_number = idx + 1
            else:
                row_number = idx + 1
            
            row_number_dict[matched_indicator] = row_number
            
            # Извлекаем значения
            for (utochnenie, ese_utochnenie), col_info in category_col_map.items():
                col_idx = col_info["col_idx"]
                graph_number = col_info["graph_number"]
                
                if col_idx < len(row):
                    value = row[col_idx]
                    
                    if pd.notna(value) and str(value).strip() not in ['', 'nan', 'NaN', 'None']:
                        try:
                            value_int = int(float(value))
                        except:
                            value_int = 0
                    else:
                        value_int = 0
                    
                    key = (matched_indicator, utochnenie, ese_utochnenie)
                    if key not in values_dict or values_dict[key]["value"] == 0:
                        values_dict[key] = {
                            "value": value_int,
                            "row_number": row_number,
                            "graph_number": graph_number
                        }
    
    # Заполняем все комбинации
    for indicator in indicator_order:
        for utochnenie, ese_utochnenie in categories:
            data = values_dict.get((indicator, utochnenie, ese_utochnenie))
            
            if data:
                value = data["value"]
                row_number = data["row_number"]
                graph_number = data["graph_number"]
            else:
                value = 0
                row_number = row_number_dict.get(indicator, 0)
                graph_number = category_col_map.get((utochnenie, ese_utochnenie), {}).get("graph_number", 0)
            
            all_data.append({
                'Показатель': indicator,
                'Уточнение': utochnenie,
                'Еще уточнение': ese_utochnenie,
                'Год': year,
                'Значение': value,
                'Строка': row_number,
                'Графа': graph_number
            })

# Создаем DataFrame
result = pd.DataFrame(all_data)

# Переупорядочиваем колонки в нужном порядке
result = result[['Показатель', 'Уточнение', 'Еще уточнение', 'Год', 'Значение', 'Строка', 'Графа']]

# Сохраняем
output_excel = r"C:\...\2300_2019-2021.xlsx"
result.to_excel(output_excel, index=False)

output_csv = r"C:\...\2300_2019-2021.csv"
result.to_csv(output_csv, index=False, encoding='utf-8-sig', sep=',')

print(f"\nГотово! Файлы сохранены:")
print(f"  Excel: {output_excel}")
print(f"  CSV: {output_csv}")
print(f"\nПример результата:")
print(result.head(15).to_string(index=False))


Обработка 2019 года...
  Найден показатель: 'Взято на учет рецидивов'
  Найден показатель: 'Взято на учет рецидивов III группы'
  Найден показатель: 'Прибыло'
  Найден показатель: 'Переведено в III группу'
  Найден показатель: 'Диагноз туберкулеза снят'
  Найден показатель: 'Выбыло'
  Найден показатель: 'Умерло от туберкулеза'
  Найден показатель: 'Умерло от других причин'

Обработка 2020 года...
  Найден показатель: 'Взято на учет рецидивов'
  Найден показатель: 'Взято на учет рецидивов III группы'
  Найден показатель: 'Прибыло'
  Найден показатель: 'Переведено в III группу'
  Найден показатель: 'Диагноз туберкулеза снят'
  Найден показатель: 'Выбыло'
  Найден показатель: 'Умерло от туберкулеза'
  Найден показатель: 'Умерло от других причин'

Обработка 2021 года...
  Найден показатель: 'Взято на учет рецидивов'
  Найден показатель: 'Взято на учет рецидивов III группы'
  Найден показатель: 'Прибыло'
  Найден показатель: 'Переведено в III группу'
  Найден показатель: 'Диагноз туберкуле